In [1]:
import json
import pandas as pd

with open("../reference/constituency_vote_ref.json", encoding="utf-8") as f:
    ref = json.load(f)

# ดู structure
print(type(ref))
if isinstance(ref, list):
    print(f"จำนวน: {len(ref)}")
    print("ตัวอย่าง:", ref[0])
elif isinstance(ref, dict):
    print("keys:", list(ref.keys())[:5])


<class 'dict'>
keys: ['province_name', 'constituency_number', 'type', 'summary_by_section', 'results']


In [5]:
print(type(ref))
print(list(ref.keys())[:5])


<class 'dict'>
['province_name', 'constituency_number', 'type', 'summary_by_section', 'results']


In [6]:
# ดู value แรก
first_key = list(ref.keys())[0]
print("key:", first_key)
print("value:", ref[first_key])


key: province_name
value: นครศรีธรรมราช


In [7]:
print("results sample:")
print(ref["results"][:2])

results sample:
[{'number': 3, 'name': 'นางสาวนันทวัน วิเชียร', 'party': 'ภูมิใจไทย', 'votes': 45334}, {'number': 1, 'name': 'นายเชาวน์วัศ เสนพงศ์', 'party': 'ประชาธิปัตย์', 'votes': 21111}]


In [3]:
# ดู candidates_ref
cand_ref = pd.read_csv("../reference/candidates_ref.csv")
print(cand_ref)


   หมายเลข                 ชื่อสกุล                 พรรค
0        1    นางสาวนันทวัน วิเชียร            ภูมิใจไทย
1        2  นายนนทิวรรธน์ นนทภักดิ์      รวมไทยสร้างชาติ
2        3     นายเชาวน์วัศ เสนพงศ์         ประชาธิปัตย์
3        4         นายเจริญ ชินวงศ์             กล้าธรรม
4        5       นายจรยุทธ มาศบำรุง              ประชาชน
5        6    นางวิไลลักษณ์ คชพันธ์             เพื่อไทย
6        7     นายปัญโน หวานนุรักษ์                พลวัต
7        8          นายทศพร แซ่ซิ้น             เศรษฐกิจ
8        9      นายสิทธิพงศ์ ชูวงศ์  สังคมประชาธิปไตยไทย
9       10       นายนพดล กาญจนรักษ์          ไทยก้าวใหม่


In [9]:
import pandas as pd

cand_ref = pd.read_csv("../reference/candidates_ref.csv")

# สร้าง mapping ชื่อ → หมายเลขของเรา
name_to_num = dict(zip(cand_ref["ชื่อสกุล"], cand_ref["หมายเลข"]))

# แปลง results เป็น DataFrame แล้ว map หมายเลข
ref_df = pd.DataFrame(ref["results"])
print(ref_df)


   number                     name                party  votes
0       3    นางสาวนันทวัน วิเชียร            ภูมิใจไทย  45334
1       1     นายเชาวน์วัศ เสนพงศ์         ประชาธิปัตย์  21111
2       5       นายจรยุทธ มาศบำรุง              ประชาชน  11586
3       4         นายเจริญ ชินวงศ์             กล้าธรรม   6989
4       2  นายนนทิวรรธน์ นนทภักดิ์      รวมไทยสร้างชาติ   4157
5       7          นายทศพร แซ่ชิ้น             เศรษฐกิจ   1077
6       6    นางวิไลลักษณ์ คชพันธ์             เพื่อไทย    842
7       8     นายปัญโน หวานนุรักษ์                พลวัต    307
8       9      นายสิทธิพงศ์ ชูวงค์  สังคมประชาธิปไตยไทย    200
9      10       นายนพดล กาญจนรักษ์          ไทยก้าวใหม่    111


In [12]:
# Map ด้วย fuzzy match เพราะชื่ออาจสะกดต่างกันนิดหน่อย
from rapidfuzz import process, fuzz

def map_name_to_our_number(json_name, name_to_num):
    # ลอง exact match ก่อน
    if json_name in name_to_num:
        return name_to_num[json_name]
    # fuzzy match
    result = process.extractOne(json_name, list(name_to_num.keys()),
                                scorer=fuzz.token_sort_ratio, score_cutoff=70)
    if result:
        return name_to_num[result[0]]
    return None

ref_df["our_number"] = ref_df["name"].apply(lambda x: map_name_to_our_number(x, name_to_num))
print(ref_df[["number", "name", "our_number", "votes"]])


   number                     name  our_number  votes
0       3    นางสาวนันทวัน วิเชียร           1  45334
1       1     นายเชาวน์วัศ เสนพงศ์           3  21111
2       5       นายจรยุทธ มาศบำรุง           5  11586
3       4         นายเจริญ ชินวงศ์           4   6989
4       2  นายนนทิวรรธน์ นนทภักดิ์           2   4157
5       7          นายทศพร แซ่ชิ้น           8   1077
6       6    นางวิไลลักษณ์ คชพันธ์           6    842
7       8     นายปัญโน หวานนุรักษ์           7    307
8       9      นายสิทธิพงศ์ ชูวงค์           9    200
9      10       นายนพดล กาญจนรักษ์          10    111


In [13]:
# โหลด OCR data
ocr_df = pd.read_csv("../output/final_csv/constituency.csv")

# รวมคะแนนต่อหมายเลข
ocr_total = ocr_df.groupby("หมายเลข")["คะแนน"].sum().reset_index()
ocr_total.columns = ["our_number", "ocr_votes"]

# Merge กับ reference
compare = ref_df[["our_number", "name", "votes"]].merge(
    ocr_total, on="our_number", how="left"
)
compare["diff"] = compare["ocr_votes"] - compare["votes"]
compare["diff_pct"] = (compare["diff"] / compare["votes"] * 100).round(2)

print(compare[["our_number", "name", "votes", "ocr_votes", "diff", "diff_pct"]])


   our_number                     name  votes  ocr_votes  diff  diff_pct
0           1    นางสาวนันทวัน วิเชียร  45334      43477 -1857     -4.10
1           3     นายเชาวน์วัศ เสนพงศ์  21111      19831 -1280     -6.06
2           5       นายจรยุทธ มาศบำรุง  11586       8745 -2841    -24.52
3           4         นายเจริญ ชินวงศ์   6989       6340  -649     -9.29
4           2  นายนนทิวรรธน์ นนทภักดิ์   4157       3757  -400     -9.62
5           8          นายทศพร แซ่ชิ้น   1077       1022   -55     -5.11
6           6    นางวิไลลักษณ์ คชพันธ์    842        703  -139    -16.51
7           7     นายปัญโน หวานนุรักษ์    307        258   -49    -15.96
8           9      นายสิทธิพงศ์ ชูวงค์    200        171   -29    -14.50
9          10       นายนพดล กาญจนรักษ์    111         96   -15    -13.51


In [17]:
ocr_df = pd.concat([
    pd.read_csv("../output/final_csv/constituency.csv"),
    pd.read_csv("../output/final_csv/17_constituency.csv"),
], ignore_index=True)

# รวมคะแนนใหม่
ocr_total = ocr_df.groupby("หมายเลข")["คะแนน"].sum().reset_index()
ocr_total.columns = ["our_number", "ocr_votes"]

compare = ref_df[["our_number", "name", "votes"]].merge(ocr_total, on="our_number", how="left")
compare["diff"] = compare["ocr_votes"] - compare["votes"]
compare["diff_pct"] = (compare["diff"] / compare["votes"] * 100).round(2)

print(compare[["our_number", "name", "votes", "ocr_votes", "diff", "diff_pct"]])


   our_number                     name  votes  ocr_votes  diff  diff_pct
0           1    นางสาวนันทวัน วิเชียร  45334      44541  -793     -1.75
1           3     นายเชาวน์วัศ เสนพงศ์  21111      20723  -388     -1.84
2           5       นายจรยุทธ มาศบำรุง  11586      11453  -133     -1.15
3           4         นายเจริญ ชินวงศ์   6989       6940   -49     -0.70
4           2  นายนนทิวรรธน์ นนทภักดิ์   4157       4105   -52     -1.25
5           8          นายทศพร แซ่ชิ้น   1077       1068    -9     -0.84
6           6    นางวิไลลักษณ์ คชพันธ์    842        835    -7     -0.83
7           7     นายปัญโน หวานนุรักษ์    307        303    -4     -1.30
8           9      นายสิทธิพงศ์ ชูวงค์    200        200     0      0.00
9          10       นายนพดล กาญจนรักษ์    111        109    -2     -1.80


In [15]:
# คะแนนรวมทั้งหมดใน OCR
ocr_grand_total = ocr_df["คะแนน"].sum()
ref_grand_total = ref_df["votes"].sum()

print(f"OCR total votes:  {ocr_grand_total:,}")
print(f"Ref total votes:  {ref_grand_total:,}")
print(f"Diff:             {ocr_grand_total - ref_grand_total:,}")
print(f"Coverage:         {ocr_grand_total / ref_grand_total * 100:.2f}%")


OCR total votes:  90,277
Ref total votes:  91,714
Diff:             -1,437
Coverage:         98.43%


In [16]:
df_main = pd.read_csv("../output/final_csv/constituency.csv")
df_17   = pd.read_csv("../output/final_csv/17_constituency.csv")

print(f"constituency.csv:    {df_main['คะแนน'].sum():,} votes, {len(df_main)} rows")
print(f"17_constituency.csv: {df_17['คะแนน'].sum():,} votes, {len(df_17)} rows")
print(f"concat total:        {ocr_df['คะแนน'].sum():,} votes, {len(ocr_df)} rows")


constituency.csv:    84,400 votes, 2560 rows
17_constituency.csv: 5,877 votes, 90 rows
concat total:        90,277 votes, 2650 rows
